# Session 7 — Epidemic signals and estimates from limited testing

**ABM course · synthetic weekly test records · unidentified pathogen**

This notebook asks three questions:

1. Do positive tests, testing volume, positivity, and trends show an epidemic-like signal?
2. How can the test data help estimate transmission \(\beta\), recovery \(\gamma\), and \(R_0\)?
3. Under a simple SIR model, when might infections decline as susceptibility falls?

The synthetic population has 10,000 people and a limit of 100 test kits per week. Week 1 uses all 100 for an all-negative baseline. In later weeks, 80 kits go to a fresh random surveillance sample and up to 20 are reserved for week-4 follow-up tests of people previously testing positive. The single CSV contains completed tests only; a follow-up row links to the same person through `person_id`.

Tests are assumed perfect: positive means currently infectious, and a negative week-4 follow-up means recovered and cleared to return to work. The notebook estimates \(\beta\) from changing positivity in random weekly samples and \(\gamma\) from observed returned week-4 retests. Since the CSV records completed tests only, it does not identify people invited who did not return or were not selected when capacity was full.

### Run instructions

Keep the CSV beside the notebook and run all cells. The notebook reads it from its GitHub raw URL. Python with pandas, NumPy, and SciPy is required.


In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import xlogy
import matplotlib.pyplot as plt
from IPython.display import display, Markdown


## 1. Load the test records

Each row is one completed test. `person_id` links a week-4 follow-up to the same person's earlier positive test. Routine surveillance tests are random samples; follow-up tests are targeted and are kept separate when estimating prevalence.


In [ ]:
TESTS_URL = "https://raw.githubusercontent.com/DACSS-Complexity/s7_epidemics/main/session7_synthetic_tests.csv"
records = pd.read_csv(TESTS_URL, parse_dates=["date"])


### Testing policy and analysis settings

- Population: 10,000; the denominator is set in the notebook, not repeated on every CSV row.
- Weekly capacity: at most 100 kits.
- Week 1: 100 random baseline tests; all are negative before the outbreak is seeded.
- Weeks 2 onward: 80 fresh random surveillance tests and up to 20 week-4 follow-up tests. Follow-ups are selected randomly if more than 20 are due. Selected people who do not return leave a kit unused; the sample is not backfilled.
- `ROLLING_WEEKS = 3` smooths the small weekly surveillance sample. `COMPARISON_WEEKS = 4` compares the first and final four observed weeks.


In [ ]:
ROLLING_WEEKS = 3
COMPARISON_WEEKS = 4
WEEKLY_KIT_LIMIT = 100


## 2. Summarize weekly testing and positivity

Use routine surveillance tests to track positivity because follow-up tests target people who were previously positive. Count all completed tests when checking weekly kit use. No week may exceed the 100-kit limit.


In [ ]:
population = 10_000
surveillance = records.loc[records.test_type.eq("surveillance")].copy()
surveillance["positive"] = surveillance.result.eq("positive").astype(int)
records["positive"] = records.result.eq("positive").astype(int)

weekly = (surveillance.groupby("week")
          .agg(date=("date", "min"), surveillance_tests=("test_id", "size"),
               positives=("positive", "sum")))
weekly["negatives"] = weekly.surveillance_tests - weekly.positives
weekly["positivity"] = weekly.positives / weekly.surveillance_tests
all_test_counts = records.groupby("week").agg(total_tests=("test_id", "size"))
weekly = weekly.join(all_test_counts)
weekly["rolling_tests"] = weekly.surveillance_tests.rolling(ROLLING_WEEKS, min_periods=1).sum()
weekly["rolling_positives"] = weekly.positives.rolling(ROLLING_WEEKS, min_periods=1).sum()
weekly["rolling_positivity"] = weekly.rolling_positives / weekly.rolling_tests

print(f"Population: {population:,}; {len(records):,} tests across {len(weekly)} weeks.")
print(f"Maximum tests in any week: {weekly.total_tests.max()} of {WEEKLY_KIT_LIMIT} kits.")
display(weekly.head(10))


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True, constrained_layout=True)
axes[0].bar(weekly.index, weekly.total_tests, color="#52677d", label="Completed tests")
axes[0].axhline(WEEKLY_KIT_LIMIT, color="#b23a48", linestyle="--", label="Weekly kit limit")
axes[0].set_ylabel("Tests per week")
axes[0].legend()
axes[1].bar(weekly.index, weekly.positives, color="#7952a3")
axes[1].set_ylabel("Positive surveillance tests")
axes[2].plot(weekly.index, 100 * weekly.positivity, color="#a7bdc5", marker="o", label="Weekly sample positivity")
axes[2].plot(weekly.index, 100 * weekly.rolling_positivity, color="#087e8b", linewidth=2,
             label=f"{ROLLING_WEEKS}-week pooled positivity")
axes[2].set_ylabel("Positive sample (%)")
axes[2].set_xlabel("Week")
axes[2].legend()
for ax in axes:
    ax.grid(alpha=.2)
plt.show()


## Question 1 — Do these tests show an epidemic signal?

Compare the first and final four weeks of the random surveillance sample. The week-4 follow-up tests are excluded from positivity because they were selected from previously positive people. A rising sample positivity with a fixed routine sample size is stronger evidence of rising prevalence than positive counts alone, although a real surveillance sample would still have sampling uncertainty.


In [ ]:
first_block = weekly.iloc[:COMPARISON_WEEKS]
last_block = weekly.iloc[-COMPARISON_WEEKS:]
def block_summary(label, frame):
    tests = int(frame.surveillance_tests.sum())
    positives = int(frame.positives.sum())
    return {"period": label, "surveillance_tests": tests,
            "positive_tests": positives,
            "pooled_positivity_pct": 100 * positives / tests if tests else np.nan,
            "average_total_kits_used": frame.total_tests.mean()}
summary = pd.DataFrame([
    block_summary(f"First {COMPARISON_WEEKS} weeks", first_block),
    block_summary(f"Final {COMPARISON_WEEKS} weeks", last_block),
])
display(summary.round(2))

first_rate = summary.iloc[0].pooled_positivity_pct
last_rate = summary.iloc[1].pooled_positivity_pct
first_positive_week = first_block.positives.mean()
last_positive_week = last_block.positives.mean()
report_q1 = (
    f"**Assessment:** The random weekly sample shows a rising epidemic-like signal. "
    f"It averaged {first_positive_week:.1f} positive tests per week in the first "
    f"{COMPARISON_WEEKS} weeks and {last_positive_week:.1f} in the final "
    f"{COMPARISON_WEEKS} weeks. Pooled sample positivity rose from {first_rate:.2f}% "
    f"to {last_rate:.2f}%. Routine surveillance used 100 tests in week 1 and 80 per "
    f"week thereafter; recorded follow-up tests used some of the remaining kits, "
    f"never exceeding the 100-kit weekly limit. The observations end "
    f"while the underlying synthetic outbreak is rising; the sample itself is small and noisy."
)
display(Markdown(report_q1))


## Question 2 — How can test data estimate \(\beta\) and \(\gamma\)?

In a homogeneous SIR model, \(\beta\) is the transmission rate per week and \(\gamma\) is the recovery rate per week. The average infectious period is \(1/\gamma\) weeks, and \(R_0 = \beta/\gamma\).

The random weekly samples estimate the infectious share of the population. We fit \(\beta\) by matching the SIR model's expected prevalence to positive counts in those samples. To estimate recovery, the code finds follow-up tests exactly four weeks after the same person's positive test. Under an exponential recovery model, the chance of still testing positive after interval \(d\) is \(e^{-\gamma d}\); positive and negative retests therefore inform \(\gamma\). Tests that are absent from the CSV do not reveal whether a person was not selected, did not return, or had recovered.


In [ ]:
from scipy.optimize import minimize_scalar

# Identify completed week-4 follow-up tests by person and interval.
prior_positive = (records.loc[records.result.eq("positive"), ["person_id", "week"]]
                 .rename(columns={"week": "positive_week"}))
returned = (records.loc[records.test_type.eq("followup")]
            .rename(columns={"week": "scheduled_week"})
            .merge(prior_positive, on="person_id", how="inner"))
returned = returned.loc[returned.scheduled_week - returned.positive_week == 4].copy()
followup_interval = (returned.scheduled_week - returned.positive_week).to_numpy(dtype=float)
followup_still_positive = returned.result.eq("positive").to_numpy(dtype=float)
followup_recovered = 1 - followup_still_positive

def recovery_negative_log_likelihood(gamma):
    survival = np.exp(-gamma * followup_interval)
    survival = np.clip(survival, 1e-12, 1 - 1e-12)
    return -np.sum(xlogy(followup_still_positive, survival) +
                   xlogy(followup_recovered, 1 - survival))

gamma_fit = minimize_scalar(recovery_negative_log_likelihood,
                            bounds=(1e-4, 2.0), method="bounded")
GAMMA_ESTIMATE = float(gamma_fit.x)

# Fit beta and the infectious count at the first post-baseline sample week.
surveillance_weekly = (surveillance.groupby("week")
                       .agg(tests=("test_id", "size"), positives=("positive", "sum")))
observed = surveillance_weekly.loc[surveillance_weekly.index > 1]
observed_weeks = observed.index.to_numpy(dtype=int)
observed_n = observed.tests.to_numpy(dtype=float)
observed_positive = observed.positives.to_numpy(dtype=float)

def sir_states_by_week(beta, initial_infectious):
    susceptible = float(population - initial_infectious)
    infectious = float(initial_infectious)
    recovered_count = 0.0
    states = []
    previous_week = int(observed_weeks[0])
    for week in observed_weeks:
        for _ in range(int(week - previous_week)):
            new_infections = susceptible * (-np.expm1(-beta * infectious / population))
            recoveries_this_week = infectious * (-np.expm1(-GAMMA_ESTIMATE))
            susceptible -= new_infections
            infectious += new_infections - recoveries_this_week
            recovered_count += recoveries_this_week
        states.append((susceptible, infectious, recovered_count))
        previous_week = int(week)
    return np.asarray(states)

def surveillance_negative_log_likelihood(log_parameters):
    beta, initial_infectious = np.exp(log_parameters)
    predicted_states = sir_states_by_week(beta, initial_infectious)
    probability_positive = np.clip(predicted_states[:, 1] / population, 1e-12, 1 - 1e-12)
    return -np.sum(xlogy(observed_positive, probability_positive) +
                   xlogy(observed_n - observed_positive, 1 - probability_positive))

initial_guess = max(1.0, observed_positive[:3].mean() / observed_n[:3].mean() * population)
fit = minimize(surveillance_negative_log_likelihood,
               x0=np.log([0.5, initial_guess]), method="L-BFGS-B",
               bounds=[(np.log(1e-3), np.log(2.0)),
                       (np.log(0.1), np.log(population - 1))])
BETA_ESTIMATE, INITIAL_INFECTIOUS_ESTIMATE = np.exp(fit.x)
R0_ESTIMATE = BETA_ESTIMATE / GAMMA_ESTIMATE
MEAN_INFECTIOUS_WEEKS_ESTIMATE = 1 / GAMMA_ESTIMATE

print(f"Population denominator: {population:,}")
print(f"Routine sample size: {int(surveillance_weekly.tests.iloc[1])} per week after baseline")
print(f"Completed week-4 follow-up tests used for gamma: {len(returned)}")
print(f"Estimated beta: {BETA_ESTIMATE:.3f} per week")
print(f"Estimated gamma: {GAMMA_ESTIMATE:.3f} per week")
print(f"Implied average infectious period: {MEAN_INFECTIOUS_WEEKS_ESTIMATE:.1f} weeks")
print(f"Estimated R0 = beta / gamma: {R0_ESTIMATE:.2f}")

fitted_states = sir_states_by_week(BETA_ESTIMATE, INITIAL_INFECTIOUS_ESTIMATE)
fitted_infectious_share = fitted_states[:, 1] / population
fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(observed_weeks, 100 * observed_positive / observed_n, color="#a7bdc5",
           label="Observed routine sample positivity")
ax.plot(observed_weeks, 100 * fitted_infectious_share, color="#087e8b", linewidth=2,
        label="Fitted SIR prevalence")
ax.set_xlabel("Week")
ax.set_ylabel("Positive sample (%)")
ax.legend()
ax.grid(alpha=.2)
plt.show()

display(Markdown(
    "**Interpretation:** weekly random tests inform prevalence and transmission; completed "
    "week-4 retests inform recovery. The CSV contains no invitation or no-show records, so "
    "missing retests cannot be interpreted as evidence of recovery."
))


## Question 3 — When might infections decline through herd immunity?

In the fitted SIR model, infections begin to decline when \(R_e(t) = R_0 S(t)/N < 1\). If immunity lasts and people mix homogeneously, this corresponds to the susceptible share falling below \(1/R_0\); the immune-share threshold is \(1 - 1/R_0\). Infections can continue for a while after that point, creating an overshoot.

The weekly test file ends during the rising phase. The date below is a conditional projection from the fitted model, not an observed downturn or a real-world forecast.


In [ ]:
# Continue the fitted weekly SIR trajectory from its final observed model state.
S_proj, I_proj, R_proj = fitted_states[-1].astype(float)
last_week = int(observed_weeks[-1])
last_date = weekly.loc[last_week, "date"]
projection_rows = []
week = last_week
while I_proj > 0 and R0_ESTIMATE * S_proj / population >= 1:
    week += 1
    new_infections = S_proj * (-np.expm1(-BETA_ESTIMATE * I_proj / population))
    recoveries_this_week = I_proj * (-np.expm1(-GAMMA_ESTIMATE))
    S_proj -= new_infections
    I_proj += new_infections - recoveries_this_week
    R_proj += recoveries_this_week
    projection_rows.append((week, last_date + pd.Timedelta(weeks=week-last_week),
                            S_proj, I_proj, R_proj, R0_ESTIMATE*S_proj/population))
projection = pd.DataFrame(projection_rows,
    columns=["week", "date", "susceptible", "infectious", "recovered", "Re"])
HERD_THRESHOLD_ESTIMATE = 1 - 1 / R0_ESTIMATE if R0_ESTIMATE > 1 else 0.0

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(observed_weeks, 100*fitted_states[:, 0]/population,
        label="Fitted susceptible (%)", color="#2878b5")
ax.plot(observed_weeks, 100*fitted_states[:, 1]/population,
        label="Fitted infectious (%)", color="#d9534f")
ax.plot(projection.week, 100*projection.susceptible/population,
        linestyle="--", color="#2878b5", label="Projected susceptible (%)")
ax.plot(projection.week, 100*projection.infectious/population,
        linestyle="--", color="#d9534f", label="Projected infectious (%)")
ax.set_xlabel("Week")
ax.set_ylabel("Population share (%)")
ax.grid(alpha=.2)
ax2 = ax.twinx()
ax2.plot(projection.week, projection.Re, color="#7952a3", linestyle=":", label=r"Projected $R_e$")
ax2.axhline(1, color="#7952a3", alpha=.5, linewidth=1)
ax2.set_ylabel(r"Effective reproduction number $R_e$")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, loc="upper right")
plt.show()

print(f"Estimated R0: {R0_ESTIMATE:.2f}; susceptible threshold 1/R0: {100/R0_ESTIMATE:.1f}%")
print(f"Implied immune-share threshold: {100*HERD_THRESHOLD_ESTIMATE:.1f}%")
if len(projection):
    crossing = projection.iloc[0]
    peak = projection.loc[projection.infectious.idxmax()]
    print(f"In this fitted scenario, Re first falls below 1 around {crossing.date.date()}.")
    print(f"The projected infectious count peaks around {peak.date.date()}.")
    print("This is conditional on the SIR and testing assumptions; no decline is observed in the sample.")


### Interpretation and limits

The synthetic exercise assumes representative weekly surveillance, perfect tests, a constant recovery hazard, lasting immunity, and homogeneous mixing. A positive person may have a week-4 retest; a negative result clears them to return to work. The single CSV includes completed tests only, so no-show and capacity non-selection rates are unavailable. Real estimates can be biased if the sample or follow-up return depends on infection status. The herd-immunity date is conditional on the fitted SIR model.


## Student exercises

1. Change `ROLLING_WEEKS` to 1 and 5. How does smoothing affect the signal in the small weekly samples?
2. Compare routine positivity with follow-up positivity. Why should follow-ups not be used as a random prevalence sample?
3. Count completed week-4 follow-up tests. How might the recovery estimate change if people still testing positive were less likely to return?
4. Increase the number of kits reserved for follow-ups and reduce routine sample size. What happens to the precision of transmission and recovery estimates?
5. Write a briefing that reports the kit policy, the estimated \(\beta\), \(\gamma\), and \(R_0\), the projected decline week, and the assumptions behind them.

### Academic takeaway

Limited weekly testing can inform an SIR model when routine tests are sampled randomly and follow-up tests track previously positive people. Routine sample positivity informs prevalence and transmission; completed week-4 retests inform recovery. Because the single CSV contains completed tests only, it cannot show why a due retest is missing. The estimates depend on those testing assumptions, and the projected decline is a model result rather than an observed event.
